In [2]:
import httpx

OLLAMA_URL = "http://localhost:11434"
# Отправляем GET-запрос и получаем ответ
response = httpx.get(f"{OLLAMA_URL}/api/tags")
# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")

Найдено моделей: 5
 - qwen2.5-coder:1.5b
 - llama3.1:8b
 - qwen2.5-coder:7b
 - lfm3:1.7b
 - qwen2.5:1.5b


In [6]:
# Готовая функция чанкинга
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [4]:
from pathlib import Path

DOCS_DIR = Path("/home/wirekit69/ai_engineer_preproject/ollama_docs")

documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}") 

Загружено документов: 70


In [7]:
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 609
Средняя длина чанка: 540 символов


In [8]:
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

--- из /home/wirekit69/ai_engineer_preproject/ollama_docs/README.md (669 символов) ---
# Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quickstart)
* [Examples](./examples.md)
* [Importing models](https://docs.ollama.com/import)
* [MacOS Documentation](https:/ ...

--- из /home/wirekit69/ai_engineer_preproject/ollama_docs/README.md (143 символов) ---
* [Troubleshooting Guide](https://docs.ollama.com/troubleshooting)
* [FAQ](https://docs.ollama.com/faq)
* [Development guide](./development.md) ...

--- из /home/wirekit69/ai_engineer_preproject/ollama_docs/api/anthropic-compatibility.mdx (226 символов) ---
---
title: Anthropic compatibility
---

Connect Anthropic clients and tools such as Claude Code to Ollama. Ollama supports a subset of the [Anthropic Messages API](https://platform.claude.com/docs/en/ ...



In [9]:
from pathlib import Path
from collections import Counter

chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

source_counts = Counter(c["source"] for c in chunks)

top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков") 

Всего чанков: 609
Файлов с чанками: 70
Топ-3 файла по числу чанков:
  /home/wirekit69/ai_engineer_preproject/ollama_docs/api.md: 74 чанков
  /home/wirekit69/ai_engineer_preproject/ollama_docs/faq.mdx: 38 чанков
  /home/wirekit69/ai_engineer_preproject/ollama_docs/capabilities/tool-calling.mdx: 33 чанков
